# A supervised machine learning model to predict personality traits, intelligence, and hireability from audio, visual, and verbal features of an Asynchronous Video Interview (AVI)

In [29]:
# load libraries
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNet ### Linear regression with combined L1/L2 regularization
from sklearn.model_selection import GridSearchCV ### Exhaustive hyperparameter search
from sklearn.model_selection import RepeatedKFold ### Repeated K-Fold cross-validation
from sklearn.model_selection import KFold ### K-Fold data splitting
from sklearn.metrics import mean_absolute_error ### MAE metric
from sklearn.metrics import mean_squared_error ### MSE regression metric
from sklearn.metrics import r2_score ### R² coefficient of determination
from sklearn.preprocessing import StandardScaler ### Standardizes features (mean=0, var=1)
from scipy.stats import pearsonr ### Pearson correlation coefficient
from sklearn.pipeline import Pipeline


In [ ]:
# Load the data
# Full data
data = pd.read_csv("input/data_gitp_short.csv")
# Advisement
AD = data.loc[
    data['VideoSimulatieType'] == 'Advisement_2',
    ['id','AD_PROBL','AD_CREAT','AD_OORDE','AD_ORGANS','AD_total']].copy()
# Change management
CM = data.loc[
    data['VideoSimulatieType'] == 'Change_Management_2',
    ['id','CM_LEIDIN', 'CM_COACH','CM_RESUL', 'CM_VISIEU','CM_total']].copy()
# Team management
TM = data.loc[
    data['VideoSimulatieType'] == 'Team_Management_2',
    ['id','TM_OVERT', 'TM_LEIDIN', 'TM_COACH','TM_RESUL','TM_total']].copy()
# Team work
TW = data.loc[
    data['VideoSimulatieType'] == 'Team_Work_2',
    ['id','TW_SAMEN', 'TW_OVERT', 'TW_MONDEC', 'TW_ORGANE','TW_total']].copy()

# load embeddings
verbal_nl = pd.read_csv("embeddings/embeddings_nl.csv")
verbal_nl_tw = pd.read_csv("embeddings/embeddings_nl_tw.csv")
verbal_en = pd.read_csv("embeddings/embeddings_en.csv")
verbal_en_tw = pd.read_csv("embeddings/embeddings_en_tw.csv")

In [31]:
# Merge competencies with embeddings
AD_embeddings = (
    AD
    .merge(verbal_nl,     on="id", how="left")
    .merge(verbal_nl_tw,  on="id", how="left")
    .merge(verbal_en,     on="id", how="left")
    .merge(verbal_en_tw,  on="id", how="left")
    .drop(columns=['Beoordeling.all_nl', 'Beoordeling.all_nl_tw', 'Beoordeling.all_en', 'Beoordeling.all_en_tw'])
)

# Change management
CM_embeddings = (
    CM
    .merge(verbal_nl,     on="id", how="left")
    .merge(verbal_nl_tw,  on="id", how="left")
    .merge(verbal_en,     on="id", how="left")
    .merge(verbal_en_tw,  on="id", how="left")
    .drop(columns=['Beoordeling.all_nl', 'Beoordeling.all_nl_tw', 'Beoordeling.all_en', 'Beoordeling.all_en_tw'])
)

# Team management
TM_embeddings = (
    TM
    .merge(verbal_nl,     on="id", how="left")
    .merge(verbal_nl_tw,  on="id", how="left")
    .merge(verbal_en,     on="id", how="left")
    .merge(verbal_en_tw,  on="id", how="left")
    .drop(columns=['Beoordeling.all_nl', 'Beoordeling.all_nl_tw', 'Beoordeling.all_en', 'Beoordeling.all_en_tw'])
)

# Team work
TW_embeddings = (
    TW
    .merge(verbal_nl,     on="id", how="left")
    .merge(verbal_nl_tw,  on="id", how="left")
    .merge(verbal_en,     on="id", how="left")
    .merge(verbal_en_tw,  on="id", how="left")
    .drop(columns=['Beoordeling.all_nl', 'Beoordeling.all_nl_tw', 'Beoordeling.all_en', 'Beoordeling.all_en_tw'])
)


In [42]:
# # verbal_nl_tw.columns
# TW_embeddings

In [33]:
# AD_embeddings.to_excel("AD_embeddings_test_for_inspection.xlsx", index=False) # export to excel for visual inspection

In [34]:
# AD_embeddings.columns

# Advisement

In [ ]:
# ==========================================
# SUPERVISED ML — AD_embeddings
# 5x10 Repeated Nested CV — Elastic Net
# ==========================================

# ------------------------
# 1. Settings
# ------------------------

RANDOM_STATE = 1337

outer_cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)

param_grid = {
    'model__alpha': np.logspace(-2, 2, 20),
    'model__l1_ratio': np.linspace(0.1, 0.9, 9)
}

# Create output directory
output_folder = "output/ML_predictions"
os.makedirs(output_folder, exist_ok=True)

# ------------------------
# 2. Define variables
# ------------------------

data = AD_embeddings.copy()

id_column = "id"

# FIX: Renamed to dependent_vars for consistency throughout script
dependent_vars = ['AD_PROBL', 'AD_CREAT', 'AD_OORDE', 'AD_ORGANS']

# AD_total exists in dataset but is NOT predicted directly
true_total = data["AD_total"]

embedding_sets = {
    "nl": [f"nl_{i}" for i in range(1024)],
    "nl_tw": [f"nl_tw_{i}" for i in range(1024)],
    "en": [f"en_{i}" for i in range(1024)],
    "en_tw": [f"en_tw_{i}" for i in range(1024)]
}

# FIX: Changed dependent_vars to dependent_vars (was undefined)
AD_predictions = data[[id_column] + dependent_vars + ["AD_total"]].copy()

performance_results = []


# ==========================================
# 3. MAIN LOOP
# ==========================================

for suffix, feature_columns in embedding_sets.items():

    print(f"\nRunning models for: {suffix}")

    X_full = data[feature_columns]

    for dv in dependent_vars:  # FIX: Changed from dependent_vars to dependent_vars

        print(f"  Predicting {dv}")

        y_full = data[dv]

        predictions_by_id = {}
        true_by_id = {}

        # ------------------------
        # OUTER CV LOOP
        # ------------------------
        for train_idx, test_idx in outer_cv.split(X_full, y_full):

            X_train = X_full.iloc[train_idx]
            X_test = X_full.iloc[test_idx]
            y_train = y_full.iloc[train_idx]
            y_test = y_full.iloc[test_idx]

            ids_test = data.iloc[test_idx][id_column]

            # Drop missing
            train_data = pd.concat([X_train, y_train], axis=1).dropna()
            test_data = pd.concat([X_test, y_test], axis=1).dropna()

            X_train_clean = train_data[feature_columns]
            y_train_clean = train_data[dv]

            X_test_clean = test_data[feature_columns]
            y_test_clean = test_data[dv]

            ids_test_clean = data.loc[test_data.index, id_column]

            # ------------------------
            # INNER CV (GridSearch)
            # ------------------------
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('model', ElasticNet(
                    random_state=RANDOM_STATE,
                    max_iter=10000,
                    tol=1e-3
                ))
            ])

            grid = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring='r2',
                n_jobs=-1,
                cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
                refit=True
            )

            grid.fit(X_train_clean, y_train_clean)

            preds = grid.best_estimator_.predict(X_test_clean)

            for i, id_val in enumerate(ids_test_clean):
                predictions_by_id[id_val] = preds[i]
                true_by_id[id_val] = y_test_clean.iloc[i]

        # ------------------------
        # Store predictions
        # ------------------------
        pred_column = f"{dv}_{suffix}"
        AD_predictions[pred_column] = np.nan

        for id_val, pred in predictions_by_id.items():
            AD_predictions.loc[
                AD_predictions[id_column] == id_val, pred_column
            ] = pred

        # ------------------------
        # Performance metrics
        # ------------------------
        ids = list(predictions_by_id.keys())

        y_true = np.array([true_by_id[i] for i in ids])
        y_pred = np.array([predictions_by_id[i] for i in ids])

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        pearson_r, p_value = pearsonr(y_true, y_pred)

        performance_results.append({
            "embedding_type": suffix,
            "variable": dv,
            "mae": round(mae, 4),
            "rmse": round(rmse, 4),
            "r2": round(r2, 4),
            "pearson_r": round(pearson_r, 4),
            "p_value": round(p_value, 6)
        })

    # ------------------------
    # Compute AD_total (Predicted)
    # ------------------------

    total_cols = [f"{dv}_{suffix}" for dv in dependent_vars]  # FIX: Changed dependent_vars to dependent_vars

    AD_predictions[f"AD_total_{suffix}"] = (
        AD_predictions[total_cols].mean(axis=1)
    )

    # ------------------------
    # Evaluate AD_total
    # ------------------------

    valid_rows = AD_predictions[
        [f"AD_total_{suffix}", "AD_total"]
    ].dropna()

    y_true_total = valid_rows["AD_total"].values
    y_pred_total = valid_rows[f"AD_total_{suffix}"].values

    mae_total = mean_absolute_error(y_true_total, y_pred_total)
    rmse_total = np.sqrt(mean_squared_error(y_true_total, y_pred_total))
    r2_total = r2_score(y_true_total, y_pred_total)
    pearson_r_total, p_total = pearsonr(y_true_total, y_pred_total)

    performance_results.append({
        "embedding_type": suffix,
        "variable": "AD_total",
        "mae": round(mae_total, 4),
        "rmse": round(rmse_total, 4),
        "r2": round(r2_total, 4),
        "pearson_r": round(pearson_r_total, 4),
        "p_value": round(p_total, 6)
    })


# ==========================================
# 4. Save outputs
# ==========================================

AD_performance = pd.DataFrame(performance_results)

AD_predictions.to_csv(
    os.path.join(output_folder, "AD_prediction.csv"),
    index=False
)

AD_predictions.to_excel(
    os.path.join(output_folder, "AD_prediction.xlsx"),
    index=False
)

AD_performance.to_csv(
    os.path.join(output_folder, "AD_performance.csv"),
    index=False
)

AD_performance.to_excel(
    os.path.join(output_folder, "AD_performance.xlsx"),
    index=False
)

print("\nDone.")


Running models for: nl
  Predicting AD_PROBL
  Predicting AD_CREAT
  Predicting AD_OORDE
  Predicting AD_ORGANS

Running models for: nl_tw
  Predicting AD_PROBL
  Predicting AD_CREAT
  Predicting AD_OORDE
  Predicting AD_ORGANS

Running models for: en
  Predicting AD_PROBL
  Predicting AD_CREAT
  Predicting AD_OORDE
  Predicting AD_ORGANS

Running models for: en_tw
  Predicting AD_PROBL
  Predicting AD_CREAT
  Predicting AD_OORDE
  Predicting AD_ORGANS

Done.


# Change management

In [38]:
# ==========================================
# SUPERVISED ML — CM_embeddings
# 5x10 Repeated Nested CV — Elastic Net
# ==========================================

# ------------------------
# 1. Settings
# ------------------------

RANDOM_STATE = 1337

outer_cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)

param_grid = {
    'model__alpha': np.logspace(-2, 2, 20),
    'model__l1_ratio': np.linspace(0.1, 0.9, 9)
}

# Create output directory
output_folder = "output/ML_predictions"
os.makedirs(output_folder, exist_ok=True)

# ------------------------
# 2. Define variables
# ------------------------

data = CM_embeddings.copy()

id_column = "id"

# FIX: Renamed to dependent_vars for consistency throughout script
dependent_vars = ['CM_LEIDIN', 'CM_COACH','CM_RESUL', 'CM_VISIEU']

# CM_total exists in dataset but is NOT predicted directly
true_total = data["CM_total"]

embedding_sets = {
    "nl": [f"nl_{i}" for i in range(1024)],
    "nl_tw": [f"nl_tw_{i}" for i in range(1024)],
    "en": [f"en_{i}" for i in range(1024)],
    "en_tw": [f"en_tw_{i}" for i in range(1024)]
}

# FIX: Changed dependent_vars to dependent_vars (was undefined)
CM_predictions = data[[id_column] + dependent_vars + ["CM_total"]].copy()

performance_results = []


# ==========================================
# 3. MAIN LOOP
# ==========================================

for suffix, feature_columns in embedding_sets.items():

    print(f"\nRunning models for: {suffix}")

    X_full = data[feature_columns]

    for dv in dependent_vars:  # FIX: Changed from dependent_vars to dependent_vars

        print(f"  Predicting {dv}")

        y_full = data[dv]

        predictions_by_id = {}
        true_by_id = {}

        # ------------------------
        # OUTER CV LOOP
        # ------------------------
        for train_idx, test_idx in outer_cv.split(X_full, y_full):

            X_train = X_full.iloc[train_idx]
            X_test = X_full.iloc[test_idx]
            y_train = y_full.iloc[train_idx]
            y_test = y_full.iloc[test_idx]

            ids_test = data.iloc[test_idx][id_column]

            # Drop missing
            train_data = pd.concat([X_train, y_train], axis=1).dropna()
            test_data = pd.concat([X_test, y_test], axis=1).dropna()

            X_train_clean = train_data[feature_columns]
            y_train_clean = train_data[dv]

            X_test_clean = test_data[feature_columns]
            y_test_clean = test_data[dv]

            ids_test_clean = data.loc[test_data.index, id_column]

            # ------------------------
            # INNER CV (GridSearch)
            # ------------------------
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('model', ElasticNet(
                    random_state=RANDOM_STATE,
                    max_iter=10000,
                    tol=1e-3
                ))
            ])

            grid = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring='r2',
                n_jobs=-1,
                cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
                refit=True
            )

            grid.fit(X_train_clean, y_train_clean)

            preds = grid.best_estimator_.predict(X_test_clean)

            for i, id_val in enumerate(ids_test_clean):
                predictions_by_id[id_val] = preds[i]
                true_by_id[id_val] = y_test_clean.iloc[i]

        # ------------------------
        # Store predictions
        # ------------------------
        pred_column = f"{dv}_{suffix}"
        CM_predictions[pred_column] = np.nan

        for id_val, pred in predictions_by_id.items():
            CM_predictions.loc[
                CM_predictions[id_column] == id_val, pred_column
            ] = pred

        # ------------------------
        # Performance metrics
        # ------------------------
        ids = list(predictions_by_id.keys())

        y_true = np.array([true_by_id[i] for i in ids])
        y_pred = np.array([predictions_by_id[i] for i in ids])

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        pearson_r, p_value = pearsonr(y_true, y_pred)

        performance_results.append({
            "embedding_type": suffix,
            "variable": dv,
            "mae": round(mae, 4),
            "rmse": round(rmse, 4),
            "r2": round(r2, 4),
            "pearson_r": round(pearson_r, 4),
            "p_value": round(p_value, 6)
        })

    # ------------------------
    # Compute CM_total (Predicted)
    # ------------------------

    total_cols = [f"{dv}_{suffix}" for dv in dependent_vars]  # FIX: Changed dependent_vars to dependent_vars

    CM_predictions[f"CM_total_{suffix}"] = (
        CM_predictions[total_cols].mean(axis=1)
    )

    # ------------------------
    # Evaluate CM_total
    # ------------------------

    valid_rows = CM_predictions[
        [f"CM_total_{suffix}", "CM_total"]
    ].dropna()

    y_true_total = valid_rows["CM_total"].values
    y_pred_total = valid_rows[f"CM_total_{suffix}"].values

    mae_total = mean_absolute_error(y_true_total, y_pred_total)
    rmse_total = np.sqrt(mean_squared_error(y_true_total, y_pred_total))
    r2_total = r2_score(y_true_total, y_pred_total)
    pearson_r_total, p_total = pearsonr(y_true_total, y_pred_total)

    performance_results.append({
        "embedding_type": suffix,
        "variable": "CM_total",
        "mae": round(mae_total, 4),
        "rmse": round(rmse_total, 4),
        "r2": round(r2_total, 4),
        "pearson_r": round(pearson_r_total, 4),
        "p_value": round(p_total, 6)
    })


# ==========================================
# 4. Save outputs
# ==========================================

CM_performance = pd.DataFrame(performance_results)

CM_predictions.to_csv(
    os.path.join(output_folder, "CM_prediction.csv"),
    index=False
)

CM_predictions.to_excel(
    os.path.join(output_folder, "CM_prediction.xlsx"),
    index=False
)

CM_performance.to_csv(
    os.path.join(output_folder, "CM_performance.csv"),
    index=False
)

CM_performance.to_excel(
    os.path.join(output_folder, "CM_performance.xlsx"),
    index=False
)

print("\nDone.")


Running models for: nl
  Predicting CM_LEIDIN
  Predicting CM_COACH
  Predicting CM_RESUL
  Predicting CM_VISIEU

Running models for: nl_tw
  Predicting CM_LEIDIN
  Predicting CM_COACH
  Predicting CM_RESUL
  Predicting CM_VISIEU

Running models for: en
  Predicting CM_LEIDIN
  Predicting CM_COACH
  Predicting CM_RESUL
  Predicting CM_VISIEU

Running models for: en_tw
  Predicting CM_LEIDIN
  Predicting CM_COACH
  Predicting CM_RESUL
  Predicting CM_VISIEU

Done.


# Team Management

In [39]:
# ==========================================
# SUPERVISED ML — TM_embeddings
# 5x10 Repeated Nested CV — Elastic Net
# ==========================================

# ------------------------
# 1. Settings
# ------------------------

RANDOM_STATE = 1337

outer_cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)

param_grid = {
    'model__alpha': np.logspace(-2, 2, 20),
    'model__l1_ratio': np.linspace(0.1, 0.9, 9)
}

# Create output directory
output_folder = "output/ML_predictions"
os.makedirs(output_folder, exist_ok=True)

# ------------------------
# 2. Define variables
# ------------------------

data = TM_embeddings.copy()

id_column = "id"

# FIX: Renamed to dependent_vars for consistency throughout script
dependent_vars = ['TM_OVERT', 'TM_LEIDIN', 'TM_COACH','TM_RESUL']

# TM_total exists in dataset but is NOT predicted directly
true_total = data["TM_total"]

embedding_sets = {
    "nl": [f"nl_{i}" for i in range(1024)],
    "nl_tw": [f"nl_tw_{i}" for i in range(1024)],
    "en": [f"en_{i}" for i in range(1024)],
    "en_tw": [f"en_tw_{i}" for i in range(1024)]
}

# FIX: Changed dependent_vars to dependent_vars (was undefined)
TM_predictions = data[[id_column] + dependent_vars + ["TM_total"]].copy()

performance_results = []


# ==========================================
# 3. MAIN LOOP
# ==========================================

for suffix, feature_columns in embedding_sets.items():

    print(f"\nRunning models for: {suffix}")

    X_full = data[feature_columns]

    for dv in dependent_vars:  # FIX: Changed from dependent_vars to dependent_vars

        print(f"  Predicting {dv}")

        y_full = data[dv]

        predictions_by_id = {}
        true_by_id = {}

        # ------------------------
        # OUTER CV LOOP
        # ------------------------
        for train_idx, test_idx in outer_cv.split(X_full, y_full):

            X_train = X_full.iloc[train_idx]
            X_test = X_full.iloc[test_idx]
            y_train = y_full.iloc[train_idx]
            y_test = y_full.iloc[test_idx]

            ids_test = data.iloc[test_idx][id_column]

            # Drop missing
            train_data = pd.concat([X_train, y_train], axis=1).dropna()
            test_data = pd.concat([X_test, y_test], axis=1).dropna()

            X_train_clean = train_data[feature_columns]
            y_train_clean = train_data[dv]

            X_test_clean = test_data[feature_columns]
            y_test_clean = test_data[dv]

            ids_test_clean = data.loc[test_data.index, id_column]

            # ------------------------
            # INNER CV (GridSearch)
            # ------------------------
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('model', ElasticNet(
                    random_state=RANDOM_STATE,
                    max_iter=10000,
                    tol=1e-3
                ))
            ])

            grid = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring='r2',
                n_jobs=-1,
                cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
                refit=True
            )

            grid.fit(X_train_clean, y_train_clean)

            preds = grid.best_estimator_.predict(X_test_clean)

            for i, id_val in enumerate(ids_test_clean):
                predictions_by_id[id_val] = preds[i]
                true_by_id[id_val] = y_test_clean.iloc[i]

        # ------------------------
        # Store predictions
        # ------------------------
        pred_column = f"{dv}_{suffix}"
        TM_predictions[pred_column] = np.nan

        for id_val, pred in predictions_by_id.items():
            TM_predictions.loc[
                TM_predictions[id_column] == id_val, pred_column
            ] = pred

        # ------------------------
        # Performance metrics
        # ------------------------
        ids = list(predictions_by_id.keys())

        y_true = np.array([true_by_id[i] for i in ids])
        y_pred = np.array([predictions_by_id[i] for i in ids])

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        pearson_r, p_value = pearsonr(y_true, y_pred)

        performance_results.append({
            "embedding_type": suffix,
            "variable": dv,
            "mae": round(mae, 4),
            "rmse": round(rmse, 4),
            "r2": round(r2, 4),
            "pearson_r": round(pearson_r, 4),
            "p_value": round(p_value, 6)
        })

    # ------------------------
    # Compute TM_total (Predicted)
    # ------------------------

    total_cols = [f"{dv}_{suffix}" for dv in dependent_vars]  # FIX: Changed dependent_vars to dependent_vars

    TM_predictions[f"TM_total_{suffix}"] = (
        TM_predictions[total_cols].mean(axis=1)
    )

    # ------------------------
    # Evaluate TM_total
    # ------------------------

    valid_rows = TM_predictions[
        [f"TM_total_{suffix}", "TM_total"]
    ].dropna()

    y_true_total = valid_rows["TM_total"].values
    y_pred_total = valid_rows[f"TM_total_{suffix}"].values

    mae_total = mean_absolute_error(y_true_total, y_pred_total)
    rmse_total = np.sqrt(mean_squared_error(y_true_total, y_pred_total))
    r2_total = r2_score(y_true_total, y_pred_total)
    pearson_r_total, p_total = pearsonr(y_true_total, y_pred_total)

    performance_results.append({
        "embedding_type": suffix,
        "variable": "TM_total",
        "mae": round(mae_total, 4),
        "rmse": round(rmse_total, 4),
        "r2": round(r2_total, 4),
        "pearson_r": round(pearson_r_total, 4),
        "p_value": round(p_total, 6)
    })


# ==========================================
# 4. Save outputs
# ==========================================

TM_performance = pd.DataFrame(performance_results)

TM_predictions.to_csv(
    os.path.join(output_folder, "TM_prediction.csv"),
    index=False
)

TM_predictions.to_excel(
    os.path.join(output_folder, "TM_prediction.xlsx"),
    index=False
)

TM_performance.to_csv(
    os.path.join(output_folder, "TM_performance.csv"),
    index=False
)

TM_performance.to_excel(
    os.path.join(output_folder, "TM_performance.xlsx"),
    index=False
)

print("\nDone.")


Running models for: nl
  Predicting TM_OVERT
  Predicting TM_LEIDIN
  Predicting TM_COACH
  Predicting TM_RESUL

Running models for: nl_tw
  Predicting TM_OVERT
  Predicting TM_LEIDIN
  Predicting TM_COACH
  Predicting TM_RESUL

Running models for: en
  Predicting TM_OVERT
  Predicting TM_LEIDIN
  Predicting TM_COACH
  Predicting TM_RESUL

Running models for: en_tw
  Predicting TM_OVERT
  Predicting TM_LEIDIN
  Predicting TM_COACH
  Predicting TM_RESUL

Done.


# Team Work

In [44]:
# ==========================================
# SUPERVISED ML — TW_embeddings
# 5x10 Repeated Nested CV — Elastic Net
# ==========================================

# ------------------------
# 1. Settings
# ------------------------

RANDOM_STATE = 1337

outer_cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)

param_grid = {
    'model__alpha': np.logspace(-2, 2, 20),
    'model__l1_ratio': np.linspace(0.1, 0.9, 9)
}

# Create output directory
output_folder = "output/ML_predictions"
os.makedirs(output_folder, exist_ok=True)

# ------------------------
# 2. Define variables
# ------------------------

data = TW_embeddings.copy()

id_column = "id"

# FIX: Renamed to dependent_vars for consistency throughout script
dependent_vars = ['TW_SAMEN', 'TW_OVERT', 'TW_MONDEC', 'TW_ORGANE']

# TW_total exists in dataset but is NOT predicted directly
true_total = data["TW_total"]

embedding_sets = {
    "nl": [f"nl_{i}" for i in range(1024)],
    "nl_tw": [f"nl_tw_{i}" for i in range(1024)],
    "en": [f"en_{i}" for i in range(1024)],
    "en_tw": [f"en_tw_{i}" for i in range(1024)]
}

# FIX: Changed dependent_vars to dependent_vars (was undefined)
TW_predictions = data[[id_column] + dependent_vars + ["TW_total"]].copy()

performance_results = []


# ==========================================
# 3. MAIN LOOP
# ==========================================

for suffix, feature_columns in embedding_sets.items():

    print(f"\nRunning models for: {suffix}")

    X_full = data[feature_columns]

    for dv in dependent_vars:  # FIX: Changed from dependent_vars to dependent_vars

        print(f"  Predicting {dv}")

        y_full = data[dv]

        predictions_by_id = {}
        true_by_id = {}

        # ------------------------
        # OUTER CV LOOP
        # ------------------------
        for train_idx, test_idx in outer_cv.split(X_full, y_full):

            X_train = X_full.iloc[train_idx]
            X_test = X_full.iloc[test_idx]
            y_train = y_full.iloc[train_idx]
            y_test = y_full.iloc[test_idx]

            ids_test = data.iloc[test_idx][id_column]

            # Drop missing
            train_data = pd.concat([X_train, y_train], axis=1).dropna()
            test_data = pd.concat([X_test, y_test], axis=1).dropna()

            X_train_clean = train_data[feature_columns]
            y_train_clean = train_data[dv]

            X_test_clean = test_data[feature_columns]
            y_test_clean = test_data[dv]

            ids_test_clean = data.loc[test_data.index, id_column]

            # ------------------------
            # INNER CV (GridSearch)
            # ------------------------
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('model', ElasticNet(
                    random_state=RANDOM_STATE,
                    max_iter=10000,
                    tol=1e-3
                ))
            ])

            grid = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring='r2',
                n_jobs=-1,
                cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
                refit=True
            )

            grid.fit(X_train_clean, y_train_clean)

            preds = grid.best_estimator_.predict(X_test_clean)

            for i, id_val in enumerate(ids_test_clean):
                predictions_by_id[id_val] = preds[i]
                true_by_id[id_val] = y_test_clean.iloc[i]

        # ------------------------
        # Store predictions
        # ------------------------
        pred_column = f"{dv}_{suffix}"
        TW_predictions[pred_column] = np.nan

        for id_val, pred in predictions_by_id.items():
            TW_predictions.loc[
                TW_predictions[id_column] == id_val, pred_column
            ] = pred

        # ------------------------
        # Performance metrics
        # ------------------------
        ids = list(predictions_by_id.keys())

        y_true = np.array([true_by_id[i] for i in ids])
        y_pred = np.array([predictions_by_id[i] for i in ids])

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        pearson_r, p_value = pearsonr(y_true, y_pred)

        performance_results.append({
            "embedding_type": suffix,
            "variable": dv,
            "mae": round(mae, 4),
            "rmse": round(rmse, 4),
            "r2": round(r2, 4),
            "pearson_r": round(pearson_r, 4),
            "p_value": round(p_value, 6)
        })

    # ------------------------
    # Compute TW_total (Predicted)
    # ------------------------

    total_cols = [f"{dv}_{suffix}" for dv in dependent_vars]  # FIX: Changed dependent_vars to dependent_vars

    TW_predictions[f"TW_total_{suffix}"] = (
        TW_predictions[total_cols].mean(axis=1)
    )

    # ------------------------
    # Evaluate TW_total
    # ------------------------

    valid_rows = TW_predictions[
        [f"TW_total_{suffix}", "TW_total"]
    ].dropna()

    y_true_total = valid_rows["TW_total"].values
    y_pred_total = valid_rows[f"TW_total_{suffix}"].values

    mae_total = mean_absolute_error(y_true_total, y_pred_total)
    rmse_total = np.sqrt(mean_squared_error(y_true_total, y_pred_total))
    r2_total = r2_score(y_true_total, y_pred_total)
    pearson_r_total, p_total = pearsonr(y_true_total, y_pred_total)

    performance_results.append({
        "embedding_type": suffix,
        "variable": "TW_total",
        "mae": round(mae_total, 4),
        "rmse": round(rmse_total, 4),
        "r2": round(r2_total, 4),
        "pearson_r": round(pearson_r_total, 4),
        "p_value": round(p_total, 6)
    })


# ==========================================
# 4. Save outputs
# ==========================================

TW_performance = pd.DataFrame(performance_results)

TW_predictions.to_csv(
    os.path.join(output_folder, "TW_prediction.csv"),
    index=False
)

TW_predictions.to_excel(
    os.path.join(output_folder, "TW_prediction.xlsx"),
    index=False
)

TW_performance.to_csv(
    os.path.join(output_folder, "TW_performance.csv"),
    index=False
)

TW_performance.to_excel(
    os.path.join(output_folder, "TW_performance.xlsx"),
    index=False
)

print("\nDone.")


Running models for: nl
  Predicting TW_SAMEN
  Predicting TW_OVERT
  Predicting TW_MONDEC
  Predicting TW_ORGANE

Running models for: nl_tw
  Predicting TW_SAMEN
  Predicting TW_OVERT
  Predicting TW_MONDEC
  Predicting TW_ORGANE

Running models for: en
  Predicting TW_SAMEN
  Predicting TW_OVERT
  Predicting TW_MONDEC
  Predicting TW_ORGANE

Running models for: en_tw
  Predicting TW_SAMEN
  Predicting TW_OVERT
  Predicting TW_MONDEC
  Predicting TW_ORGANE

Done.
